# 02 – SQL Basics with PostgreSQL

You learn `SELECT`, `WHERE`, `ORDER BY`, `GROUP BY` and `JOIN` using the Northwind database.

> **SOLUTION** – try the exercises yourself first!

In [ ]:
# Run this cell first! It loads the helper functions.
import os
if os.path.isdir("Exercises"):          # started in the project folder
    os.chdir("Exercises")
elif not os.path.exists("helpers.py"):  # started in the solutions folder
    os.chdir("..")
from helpers import *

# Load the Northwind data into PostgreSQL (only needed once)
setup_database()

## The Northwind database

Northwind is a small shop database. The tables:

| Table | What is in it | Important columns |
|---|---|---|
| `customers` | customers | customerid, customername, city, country |
| `orders` | orders | orderid, customerid, employeeid, orderdate, shipperid |
| `orderdetails` | products in each order | orderid, productid, quantity |
| `products` | products | productid, productname, supplierid, categoryid, price |
| `categories` | product categories | categoryid, categoryname |
| `employees` | employees | employeeid, firstname, lastname |
| `shippers` | delivery companies | shipperid, shippername |
| `suppliers` | suppliers | supplierid, suppliername, country |

## How these exercises work

`sql("...")` runs a query and shows the result as a table. Try it:

In [ ]:
sql("SELECT * FROM categories")

In each exercise, write your SQL between the `"""` quotes and run the cell.
`check_sql()` shows your result **and** tells you if it is correct.

💡 You can also run the same queries in **pgAdmin** (http://localhost:5050 →
Servers → PostgreSQL → Databases → northwind → right click → *Query Tool*).

---
## Part 1: SELECT and FROM

**SQL** (Structured Query Language) is the language to ask a database for data.
The basic form is:

```sql
SELECT column1, column2   -- which columns?
FROM table_name           -- which table?
```
`SELECT *` means *all columns*.

### Exercise 1

Show all columns and rows of the table `shippers`.

In [ ]:
query = """
SELECT * FROM shippers
"""
check_sql(1, query)

### Exercise 2

Show only the columns `productname` and `price` of all `products`.

In [ ]:
query = """
SELECT productname, price
FROM products
"""
check_sql(2, query)

---
## Part 2: Filtering with WHERE

`WHERE` keeps only the rows that match a condition:

```sql
SELECT * FROM products WHERE price < 10
SELECT * FROM customers WHERE city = 'London'   -- text in 'single quotes'!
```
Operators: `=`, `<>` (not equal), `>`, `<`, `>=`, `<=`, and combine with `AND` / `OR`.

### Exercise 3

Show all columns of the customers from `'Germany'`.

In [ ]:
query = """
SELECT *
FROM customers
WHERE country = 'Germany'
"""
check_sql(3, query)

### Exercise 4

Show `productname` and `price` of all products with a price **greater than 50**.

In [ ]:
query = """
SELECT productname, price
FROM products
WHERE price > 50
"""
check_sql(4, query)

### Exercise 5

Show `productname` and `price` of all products in category 1 (`categoryid = 1`) that cost **less than 20**.

In [ ]:
query = """
SELECT productname, price
FROM products
WHERE categoryid = 1 AND price < 20
"""
check_sql(5, query)

### Exercise 6

Show `customername` and `country` of all customers from France, Spain or Italy.

Tip: `WHERE country IN ('A', 'B', 'C')` is shorter than many `OR`s.

In [ ]:
query = """
SELECT customername, country
FROM customers
WHERE country IN ('France', 'Spain', 'Italy')
"""
check_sql(6, query)

### Exercise 7

Show the `productname` of all products whose name **starts with** `Ch`.

Tip: `LIKE 'Ch%'` – the `%` means *any text*.

In [ ]:
query = """
SELECT productname
FROM products
WHERE productname LIKE 'Ch%'
"""
check_sql(7, query)

---
## Part 3: Sorting and limiting

```sql
SELECT * FROM products ORDER BY price DESC   -- DESC = big to small, ASC = small to big
SELECT * FROM products LIMIT 3               -- only the first 3 rows
SELECT DISTINCT city FROM customers          -- no duplicates
```

### Exercise 8

Show `productname` and `price` of all products, the most expensive first.

In [ ]:
query = """
SELECT productname, price
FROM products
ORDER BY price DESC
"""
check_sql(8, query)

### Exercise 9

Show `productname` and `price` of the **5 cheapest** products.

In [ ]:
query = """
SELECT productname, price
FROM products
ORDER BY price ASC
LIMIT 5
"""
check_sql(9, query)

### Exercise 10

Show every `country` of the customers only **once**, sorted A to Z.

In [ ]:
query = """
SELECT DISTINCT country
FROM customers
ORDER BY country
"""
check_sql(10, query)

---
## Part 4: Aggregate functions

Aggregate functions calculate **one value from many rows**:
`COUNT(*)`, `SUM(col)`, `AVG(col)`, `MIN(col)`, `MAX(col)`.

```sql
SELECT COUNT(*) FROM orders
SELECT ROUND(AVG(price)::numeric, 2) FROM products   -- round to 2 decimals
```
(`::numeric` converts the value so PostgreSQL can round it.)

### Exercise 11

How many customers are there?

In [ ]:
query = """
SELECT COUNT(*)
FROM customers
"""
check_sql(11, query)

### Exercise 12

Show the lowest price, the highest price and the average price (rounded to 2 decimals) of all products – in this order, in one row.

In [ ]:
query = """
SELECT MIN(price), MAX(price), ROUND(AVG(price)::numeric, 2)
FROM products
"""
check_sql(12, query)

---
## Part 5: GROUP BY and HAVING

`GROUP BY` puts rows with the same value into groups, and an aggregate
function is calculated **per group** (like the counting dictionary in the
Python notebook!):

```sql
SELECT categoryid, COUNT(*) AS number_of_products   -- AS gives a column a name
FROM products
GROUP BY categoryid
```
`HAVING` filters **groups** (`WHERE` filters **rows** before grouping).

### Exercise 13

Show each `country` and the number of customers in it. Sort by the number **descending**, and if equal, by country A to Z.

Tip: `ORDER BY number_of_customers DESC, country`

In [ ]:
query = """
SELECT country, COUNT(*) AS number_of_customers
FROM customers
GROUP BY country
ORDER BY number_of_customers DESC, country
"""
check_sql(13, query)

### Exercise 14

Show `country` and the number of customers, but only countries with **more than 5** customers.

In [ ]:
query = """
SELECT country, COUNT(*)
FROM customers
GROUP BY country
HAVING COUNT(*) > 5
"""
check_sql(14, query)

---
## Part 6: JOIN – combining tables

Data is split into several tables. `products` only has a `categoryid`, the
name of the category is in `categories`. A `JOIN` connects them:

```sql
SELECT o.orderid, s.shippername
FROM orders o                               -- o is a short name (alias)
JOIN shippers s ON o.shipperid = s.shipperid
```

### Exercise 15

Show `productname` and `categoryname` for all products.

In [ ]:
query = """
SELECT p.productname, c.categoryname
FROM products p
JOIN categories c ON p.categoryid = c.categoryid
"""
check_sql(15, query)

### Exercise 16

Show every `categoryname` and how many products it has. Sort by the number descending, and if equal, by category name.

In [ ]:
query = """
SELECT c.categoryname, COUNT(*) AS number_of_products
FROM products p
JOIN categories c ON p.categoryid = c.categoryid
GROUP BY c.categoryname
ORDER BY number_of_products DESC, c.categoryname
"""
check_sql(16, query)

### Exercise 17

Show the **top 5 customers** (`customername`) with the most orders and their number of orders. If equal, sort by customer name.

In [ ]:
query = """
SELECT c.customername, COUNT(*) AS number_of_orders
FROM customers c
JOIN orders o ON c.customerid = o.customerid
GROUP BY c.customername
ORDER BY number_of_orders DESC, c.customername
LIMIT 5
"""
check_sql(17, query)

### Exercise 18

Show `firstname`, `lastname` of each employee and how many orders they handled, the most first.

In [ ]:
query = """
SELECT e.firstname, e.lastname, COUNT(*) AS number_of_orders
FROM employees e
JOIN orders o ON e.employeeid = o.employeeid
GROUP BY e.firstname, e.lastname
ORDER BY number_of_orders DESC
"""
check_sql(18, query)

---
🎉 **Great job!** Continue with **`03_python_and_sql.ipynb`**.

Want more? The file `SQL/QUESTIONS.sql` has harder questions
(solutions in `SQL/QUESTIONS_SOLUTION.sql`).